# Anomaly Detection in Images Using Convolutional Autoencoders

**Dataset:** MVTec AD, category `screw`  |  **Model:** CNN autoencoder (TensorFlow/Keras)

The autoencoder is trained only on normal screw images. At test time
the reconstruction error decides NORMAL vs ANOMALY:

    error_map = mean over channels of (original - reconstruction)^2
    score     = mean of the error map
    threshold = mean + 3 * std of the scores of normal images

The encoder ends at a 16x16x16 bottleneck. With 4,096 latent values
for 16,384 input pixels the network is forced to compress, so it
cannot simply copy a defect into the output.

Two threshold protocols are shown:
1. the validation-split threshold from `train.py`
2. an in-domain threshold calibrated on held-out normal test images

> Before running: **Runtime -> Change runtime type -> GPU**.

In [ ]:
# Cell 1 - Install dependencies (Colab already has most of these)
!pip -q install tensorflow opencv-python scikit-learn matplotlib pillow
print('Dependencies ready.')

In [ ]:
# Cell 2 - Check GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPUs detected:', gpus)
if not gpus:
    print('WARNING: no GPU. Runtime > Change runtime type > GPU.')

In [ ]:
# Cell 3 - Dataset path
from pathlib import Path

DATASET_ROOT = Path('mvtec_anomaly_detection')
if (DATASET_ROOT / 'screw' / 'train' / 'good').is_dir():
    print('Dataset found:', DATASET_ROOT.resolve())
else:
    print('Dataset not found yet - run Cell 4 to upload and extract it.')

In [ ]:
# Cell 4 - Upload the MVTec archive and extract it
from google.colab import files

if not Path('mvtec_anomaly_detection/screw').is_dir():
    uploaded = files.upload()  # choose mvtec_anomaly_detection.tar.xz
    archive = next(iter(uploaded))
    print('Extracting (this takes a few minutes)...')
    !tar -xJf {archive}
assert (DATASET_ROOT / 'screw' / 'train' / 'good').is_dir()
print('Dataset ready.')

In [ ]:
# Cell 5 - Create anomaly_detection.py (core pipeline)
%%writefile anomaly_detection.py
"""
Anomaly Detection in Images Using Convolutional Autoencoders
=============================================================
MVTec AD dataset. The default category is "screw"; the scripts also accept
`--category wood` and `--category bottle`.

The autoencoder is trained only on normal (good) images of one category. At test time
the image is reconstructed and the reconstruction error is compared with a
threshold learnt from normal images:

    error_map       = mean over channels of (original - reconstructed)^2
    anomaly_score   = mean of the error map
    prediction      = ANOMALY if anomaly_score > threshold else NORMAL

The threshold is computed as  mean + 3 * std  of the errors on normal
validation images (the same formula taught in class). No test image and no
defect label is used while training or while fixing the threshold.
"""

import json
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

# ---------------------------------------------------------------------------
# Settings (change these here; every script imports them)
# ---------------------------------------------------------------------------

DATASET_ROOT = Path("mvtec_anomaly_detection")
CATEGORY = "screw"  # default; train/evaluate/predict override it with --category

IMAGE_SIZE = 128
CHANNELS = 1

BATCH_SIZE = 32
EPOCHS = 50
VALIDATION_SPLIT = 0.2
EARLY_STOP_PATIENCE = 10

# BatchNormalization keeps running estimates of mean/variance that are only
# used at prediction time. On the first few epochs those estimates are still
# near their defaults, so the validation loss looks terrible even while the
# training loss falls, and EarlyStopping would freeze an unusable model
# (this happened on the bottle category). Ignore validation loss until the
# estimates have had a few epochs to settle.
EARLY_STOP_START = 5

# How fast BatchNormalization forgets its initial mean=0 / variance=1 guess.
# The Keras default (0.99) needs ~500 batches to converge; with 200 training
# images that is 30 epochs, so predictions would use badly wrong statistics
# and the reconstruction collapses to a constant image. 0.9 converges in
# about 30 batches (a few epochs).
BATCHNORM_MOMENTUM = 0.9

# Light augmentation of the NORMAL training images only (flips, brightness,
# contrast, a little noise). Validation and test images are never augmented.
# This stops the autoencoder from memorising the training pictures; it has to
# learn the general structure of a normal part instead.
AUGMENT = True
AUGMENT_BRIGHTNESS = 0.08
AUGMENT_NOISE_STD = 0.01

# Width of the latent (bottleneck) feature map. At 128x128 the encoder ends
# at 16x16, so 16 filters give 16*16*16 = 4096 latent values for 16384 input
# pixels. The network is forced to compress, which is what makes it fail on
# defects instead of copying them. A wider bottleneck (e.g. 128 filters)
# gives more latent values than the image has pixels, and the model simply
# memorises the input.
LATENT_FILTERS = 16

SEED = 42

MODELS_DIR = Path("models")
RESULTS_ROOT = Path("results")


# Every category gets its own model files and its own results folder, so
# training wood does not overwrite the screw model or its figures.

def category_paths(category=CATEGORY):
    """All input and output paths that belong to one MVTec category."""
    results = RESULTS_ROOT / category
    return {
        "model": MODELS_DIR / f"{category}_autoencoder.keras",
        "threshold": MODELS_DIR / f"{category}_threshold.json",
        "calibrated_threshold":
            MODELS_DIR / f"{category}_threshold_calibrated.json",
        "results": results,
        "heatmaps": results / "heatmaps",
        "train_good": DATASET_ROOT / category / "train" / "good",
        "test": DATASET_ROOT / category / "test",
        "ground_truth": DATASET_ROOT / category / "ground_truth",
    }


# Defaults = the screw category, used by the web app and by predict.py.
PATHS = category_paths()

MODEL_PATH = PATHS["model"]
THRESHOLD_PATH = PATHS["threshold"]
CALIBRATED_THRESHOLD_PATH = PATHS["calibrated_threshold"]

RESULTS_DIR = PATHS["results"]
HEATMAPS_DIR = PATHS["heatmaps"]

TRAIN_GOOD_DIR = PATHS["train_good"]
TEST_DIR = PATHS["test"]
GROUND_TRUTH_DIR = PATHS["ground_truth"]

# mean_std -> mean + n_std * std, percentile -> given percentile, max -> largest
THRESHOLD_STRATEGY = "mean_std"
THRESHOLD_N_STD = 3.0
THRESHOLD_PERCENTILE = 99.0
MASK_N_STD = 3.0
MASK_PERCENTILE = 99.0


# ---------------------------------------------------------------------------
# Reproducibility and dataset checks
# ---------------------------------------------------------------------------


def set_seeds(seed=SEED):
    """Fix the random seeds so a run can be reproduced."""
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def check_gpu():
    """Print the GPUs TensorFlow can see (useful in Colab)."""
    gpus = tf.config.list_physical_devices("GPU")
    print("TensorFlow version:", tf.__version__)
    if gpus:
        for gpu in gpus:
            print("GPU:", gpu)
    else:
        print("No GPU found, training will run on CPU.")
    return gpus


def check_dataset(category=CATEGORY):
    """Stop early with a clear message if the MVTec folder is missing."""
    paths = category_paths(category)
    if not DATASET_ROOT.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {DATASET_ROOT.resolve()}"
        )
    if not paths["train_good"].exists():
        raise FileNotFoundError(
            f"[{category}] Missing normal training images: {paths['train_good']}"
        )
    if not paths["test"].exists():
        raise FileNotFoundError(
            f"[{category}] Missing test folder: {paths['test']}"
        )



# Image loading and preprocessing


def preprocess_image(image):
 
    img = image.astype(np.float32)
    if img.max() > 1.0:
        img = img / 255.0
    img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
    img = np.clip(img, 0.0, 1.0)
    return img[..., np.newaxis]  # (IMAGE_SIZE, IMAGE_SIZE, 1)


def load_image(image_path):
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return preprocess_image(image)


def load_mask(mask_path):

    path = Path(mask_path)
    if not path.exists():
        raise FileNotFoundError(f"Mask not found: {path}")

    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Could not read mask: {path}")
    mask = cv2.resize(mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)
    return mask > 127


def _image_files(folder):

    files = sorted(Path(folder).glob("*.png"))
    if not files:
        raise FileNotFoundError(f"No .png images in {folder}")
    return files


def load_normal_images(category=CATEGORY):

    check_dataset(category)

    paths = _image_files(category_paths(category)["train_good"])
    print(f"[{category}] Normal training images: {len(paths)}")

    images = np.stack([load_image(p) for p in paths]).astype(np.float32)

    x_train, x_val = train_test_split(
        images, test_size=VALIDATION_SPLIT, random_state=SEED, shuffle=True
    )
    print(f"Split: {len(x_train)} train, {len(x_val)} validation")
    return x_train, x_val


def load_test_images(category=CATEGORY):

    check_dataset(category)
    paths = category_paths(category)

    images = []
    labels = []
    defect_types = []
    mask_paths = []

    for defect_dir in sorted(d for d in paths["test"].iterdir() if d.is_dir()):
        name = defect_dir.name
        files = sorted(defect_dir.glob("*.png"))
        print(f"test/{name}: {len(files)} images")

        for path in files:
            images.append(load_image(path))
            labels.append(0 if name == "good" else 1)
            defect_types.append(name)

            if name == "good":
                mask_paths.append(None)
            else:
                mask_paths.append(
                    paths["ground_truth"] / name / f"{path.stem}_mask.png"
                )

    return (
        np.stack(images).astype(np.float32),
        np.array(labels),
        defect_types,
        mask_paths,
    )

# Model

def build_autoencoder():
    
    inp = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))

    # Encoder: 128 -> 64 -> 32 -> 16
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(inp)
    x = layers.BatchNormalization(momentum=BATCHNORM_MOMENTUM)(x)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization(momentum=BATCHNORM_MOMENTUM)(x)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization(momentum=BATCHNORM_MOMENTUM)(x)
    x = layers.MaxPool2D()(x)

    # bottleneck
    x = layers.Conv2D(LATENT_FILTERS, 3, activation="relu", padding="same")(x)

    # Decoder: 16 -> 32 -> 64 -> 128
    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.BatchNormalization(momentum=BATCHNORM_MOMENTUM)(x)
    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.BatchNormalization(momentum=BATCHNORM_MOMENTUM)(x)
    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.BatchNormalization(momentum=BATCHNORM_MOMENTUM)(x)

    out = layers.Conv2D(CHANNELS, 3, activation="sigmoid", padding="same")(x)

    model = models.Model(inp, out, name="screw_autoencoder")
    model.compile(optimizer="adam", loss="mse")
    return model


def describe_architecture(model):
    """Print the layer table and the size of the bottleneck."""
    latent_size = IMAGE_SIZE // 8
    values = latent_size * latent_size * LATENT_FILTERS
    print("=" * 60)
    print("CONVOLUTIONAL AUTOENCODER")
    print(f"Latent shape: {latent_size} x {latent_size} x {LATENT_FILTERS} "
          f"= {values} values")
    print("=" * 60)
    model.summary()


# Training

def _augment_dataset(images):

    dataset = tf.data.Dataset.from_tensor_slices(images.astype(np.float32))
    dataset = dataset.shuffle(len(images), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(BATCH_SIZE)

    def augment(batch):
        x = tf.image.random_flip_left_right(batch)
        x = tf.image.random_flip_up_down(x)
        x = tf.image.random_brightness(x, max_delta=AUGMENT_BRIGHTNESS)
        x = tf.image.random_contrast(x, lower=0.9, upper=1.1)
        x = x + tf.random.normal(tf.shape(x), stddev=AUGMENT_NOISE_STD)
        x = tf.clip_by_value(x, 0.0, 1.0)
        return x, x

    dataset = dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.prefetch(tf.data.AUTOTUNE)


def train_autoencoder(model, x_train, x_val):
    """Train the autoencoder to reproduce its input (x -> x)."""
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOP_PATIENCE,
        start_from_epoch=EARLY_STOP_START,
        restore_best_weights=True,
        verbose=1,
    )
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    )
    callbacks = [early_stop, reduce_lr]

    if AUGMENT:
        print("Augmentation ON (normal training images only)")
        history = model.fit(
            _augment_dataset(x_train),
            validation_data=(x_val, x_val),
            epochs=EPOCHS,
            callbacks=callbacks,
            verbose=2,
        )
    else:
        history = model.fit(
            x_train,
            x_train,
            validation_data=(x_val, x_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=callbacks,
            shuffle=True,
            verbose=2,
        )
    return history


def plot_training_history(history, save_path):
    """Save the training and validation loss curves."""
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], marker="o", label="Training loss")
    plt.plot(history.history["val_loss"], marker="s", label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss")
    plt.title("Training vs Validation Loss")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print("Saved training curve:", save_path)


# ---------------------------------------------------------------------------
# Reconstruction error and anomaly score
# ---------------------------------------------------------------------------


def reconstruct(model, image):
    """Reconstruct one (H, W, 1) image."""
    image = np.asarray(image, dtype=np.float32)
    if image.ndim == 2:
        image = image[..., np.newaxis]
    if image.ndim == 3:
        image = image[np.newaxis, ...]

    reconstruction = model.predict(image, verbose=0)
    return reconstruction[0]


def reconstruction_error(original, reconstructed):
    """Pixel-wise squared error, averaged over channels -> (H, W) map."""
    difference = original.astype(np.float32) - reconstructed.astype(np.float32)
    return np.mean(np.square(difference), axis=-1)


def anomaly_score(error_map):
    """One number per image: the mean of its error map."""
    return float(np.mean(error_map))


def score_images(model, images, batch_size=BATCH_SIZE):
    """Score many images in batches.

    Returns scores (N,) and error_maps (N, H, W).
    """
    scores = np.zeros(len(images), dtype=np.float64)
    error_maps = np.zeros(images.shape[:3], dtype=np.float32)

    for start in range(0, len(images), batch_size):
        batch = images[start:start + batch_size]
        reconstructions = model.predict(batch, verbose=0)
        errors = np.mean(np.square(batch - reconstructions), axis=-1)

        error_maps[start:start + len(batch)] = errors
        scores[start:start + len(batch)] = errors.mean(axis=(1, 2))

    return scores, error_maps


# ---------------------------------------------------------------------------
# Thresholds (computed from normal images only)
# ---------------------------------------------------------------------------


def compute_threshold(normal_scores, strategy=THRESHOLD_STRATEGY,
                      n_std=THRESHOLD_N_STD, percentile=THRESHOLD_PERCENTILE):
    """Image-level threshold from the scores of normal images.

    mean_std   : mean + n_std * std        (the formula used in class)
    percentile : e.g. the 99th percentile
    max        : the largest normal score
    """
    scores = np.asarray(normal_scores, dtype=np.float64)
    mean_error = float(scores.mean())
    std_error = float(scores.std())

    if strategy == "mean_std":
        threshold = mean_error + n_std * std_error
    elif strategy == "percentile":
        threshold = float(np.percentile(scores, percentile))
    elif strategy == "max":
        threshold = float(scores.max())
    else:
        raise ValueError("strategy must be 'mean_std', 'percentile' or 'max'")

    return {
        "threshold": threshold,
        "strategy": strategy,
        "mean_normal_error": mean_error,
        "std_normal_error": std_error,
        "n_std": n_std,
        "percentile": percentile,
    }


def compute_pixel_threshold(normal_error_maps, strategy=THRESHOLD_STRATEGY,
                            n_std=MASK_N_STD, percentile=MASK_PERCENTILE):
    """Pixel-level threshold from the error maps of normal images."""
    pixels = np.concatenate([m.ravel() for m in normal_error_maps])

    if strategy == "mean_std":
        return float(pixels.mean() + n_std * pixels.std())
    if strategy == "percentile":
        return float(np.percentile(pixels, percentile))
    if strategy == "max":
        return float(pixels.max())
    raise ValueError("strategy must be 'mean_std', 'percentile' or 'max'")


def calibrate_threshold(model, normal_images, category=CATEGORY):
    """Fit the threshold on normal images from the deployment batch.

    train/good and test/good come from different captures, so a threshold
    fitted on train/good can be too high for test/good (it then flags
    nothing). Here the reference images come from the same batch that will
    be scored, which is what a real line-side calibration would do.
    """
    scores, error_maps = score_images(model, normal_images)

    info = compute_threshold(scores)
    info["category"] = category
    info["calibration"] = "in_domain_normal_images"
    info["pixel_threshold"] = compute_pixel_threshold(error_maps)
    info["n_calibration_images"] = int(len(normal_images))
    return info


def make_mask(error_map, pixel_threshold):
    """Turn an error map into a binary mask and remove single speckles."""
    mask = (error_map > pixel_threshold).astype(np.uint8)
    mask = cv2.medianBlur(mask, 3)
    return mask.astype(bool)


# ---------------------------------------------------------------------------
# Save and load
# ---------------------------------------------------------------------------


def save_model(model, path=MODEL_PATH):
    """Save architecture, weights and compile settings in one .keras file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    model.save(path)
    print("Saved model:", path)
    return path


def load_model(path=MODEL_PATH):
    """Load a saved model without retraining."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Model not found: {path}. Run train.py first.")
    print("Loaded model:", path)
    return models.load_model(path)


def save_threshold(info, path=THRESHOLD_PATH):
    """Save the threshold dictionary as JSON."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=4)
    print("Saved threshold:", path)
    return path


def load_threshold(path=THRESHOLD_PATH):
    """Load a threshold JSON file."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Threshold not found: {path}. Run train.py first.")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ---------------------------------------------------------------------------
# Prediction
# ---------------------------------------------------------------------------


def predict(model, image_path, threshold, pixel_threshold=None):
    """Full prediction for one image.

    Returns a dictionary with the prediction, the score, the reconstruction
    and the error map (plus a mask when a pixel threshold is given).
    """
    original = load_image(image_path)
    reconstructed = reconstruct(model, original)
    error_map = reconstruction_error(original, reconstructed)
    score = anomaly_score(error_map)

    result = {
        "image_path": str(image_path),
        "prediction": "ANOMALY" if score > threshold else "NORMAL",
        "anomaly_score": score,
        "threshold": float(threshold),
        "original": original,
        "reconstruction": reconstructed,
        "error_map": error_map,
    }

    if pixel_threshold is not None:
        result["predicted_mask"] = make_mask(error_map, pixel_threshold)

    return result


# ---------------------------------------------------------------------------
# Visualisation
# ---------------------------------------------------------------------------


def heatmap_overlay(original, error_map):
    """Blend a JET colour map of the error over the grayscale image."""
    error = error_map / max(float(error_map.max()), 1e-12)
    heat = cv2.applyColorMap((error * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)

    base = (original.squeeze() * 255).astype(np.uint8)
    base = cv2.cvtColor(base, cv2.COLOR_GRAY2RGB)

    return cv2.addWeighted(base, 0.6, heat, 0.4, 0)


def show_heatmap(result, save_path=None, show=False, title_prefix=""):
    """2x2 figure: original | reconstruction // error map | overlay."""
    overlay = heatmap_overlay(result["original"], result["error_map"])

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    axes[0, 0].imshow(result["original"].squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[0, 0].set_title("Original")
    axes[0, 1].imshow(result["reconstruction"].squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[0, 1].set_title("Reconstruction")
    image = axes[1, 0].imshow(result["error_map"], cmap="hot")
    axes[1, 0].set_title("Error map")
    fig.colorbar(image, ax=axes[1, 0], fraction=0.046)
    axes[1, 1].imshow(overlay)
    axes[1, 1].set_title("Heatmap overlay")

    for ax in axes.ravel():
        ax.axis("off")

    fig.suptitle(
        f"{title_prefix}{result['prediction']}   "
        f"score {result['anomaly_score']:.4f}   "
        f"threshold {result['threshold']:.4f}"
    )
    fig.tight_layout(rect=(0, 0, 1, 0.96))

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
        print("Saved heatmap:", save_path)
    if show:
        plt.show()
    plt.close(fig)


def show_localization(result, gt_mask, save_path=None):
    """2x2 figure: original | ground truth // predicted mask | overlay."""
    overlay = heatmap_overlay(result["original"], result["error_map"])
    predicted_mask = result.get("predicted_mask")

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    axes[0, 0].imshow(result["original"].squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[0, 0].set_title("Original")
    axes[0, 1].imshow(gt_mask, cmap="gray")
    axes[0, 1].set_title("Ground-truth mask")
    axes[1, 0].imshow(predicted_mask, cmap="gray")
    axes[1, 0].set_title("Predicted mask")
    axes[1, 1].imshow(overlay)
    axes[1, 1].set_title("Heatmap overlay")

    for ax in axes.ravel():
        ax.axis("off")

    fig.suptitle("Localization: predicted mask vs ground truth")
    fig.tight_layout(rect=(0, 0, 1, 0.96))

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150)
        print("Saved localization figure:", save_path)
    plt.close(fig)


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------


def evaluate_image_level(model, images, labels, threshold, batch_size=BATCH_SIZE):
    """Accuracy, precision, recall, F1 and ROC-AUC on the test set.

    ROC-AUC uses the continuous scores; the other metrics use the
    thresholded predictions.
    """
    scores, error_maps = score_images(model, images, batch_size)
    y_true = np.asarray(labels, dtype=np.int64)
    y_pred = (scores > threshold).astype(np.int64)

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "y_true": y_true,
        "y_pred": y_pred,
        "scores": scores,
        "error_maps": error_maps,
    }

    fpr, tpr, _ = roc_curve(y_true, scores)
    metrics["fpr"] = fpr
    metrics["tpr"] = tpr
    return metrics


def evaluate_localization(model, images, mask_paths, pixel_threshold,
                          batch_size=BATCH_SIZE):
    """Pixel-level IoU / precision / recall / F1 against the MVTec masks."""
    tp = fp = fn = 0
    defect_count = 0

    for start in range(0, len(images), batch_size):
        batch = images[start:start + batch_size]
        reconstructions = model.predict(batch, verbose=0)
        errors = np.mean(np.square(batch - reconstructions), axis=-1)

        for i, error_map in enumerate(errors):
            index = start + i
            if mask_paths[index] is None:
                continue

            gt_mask = load_mask(mask_paths[index])
            predicted_mask = make_mask(error_map, pixel_threshold)

            tp += int(np.sum(predicted_mask & gt_mask))
            fp += int(np.sum(predicted_mask & ~gt_mask))
            fn += int(np.sum(~predicted_mask & gt_mask))
            defect_count += 1

    iou = tp / (tp + fp + fn) if (tp + fp + fn) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0

    return {
        "n_defect_images": defect_count,
        "iou": iou,
        "pixel_precision": precision,
        "pixel_recall": recall,
        "pixel_f1": f1,
    }


def plot_confusion_matrix(cm, save_path):
    """Save the confusion matrix as a figure."""
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm, cmap="Blues")
    fig.colorbar(image, ax=ax)

    labels = ["NORMAL", "ANOMALY"]
    ax.set_xticks([0, 1], labels)
    ax.set_yticks([0, 1], labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title("Confusion Matrix")

    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print("Saved confusion matrix:", save_path)


def plot_roc_curve(fpr, tpr, roc_auc, save_path):
    """Save the ROC curve."""
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, linewidth=2, label=f"ROC (AUC = {roc_auc:.4f})")
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Chance")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print("Saved ROC curve:", save_path)


def print_report(image_metrics, localization_metrics, threshold):
    """Print the evaluation results in a readable form."""
    print("=" * 60)
    print("IMAGE-LEVEL EVALUATION")
    print("=" * 60)
    print(f"Threshold        : {threshold:.6f}")
    print(f"Accuracy         : {image_metrics['accuracy']:.4f}")
    print(f"Precision        : {image_metrics['precision']:.4f}")
    print(f"Recall           : {image_metrics['recall']:.4f}")
    print(f"F1-score         : {image_metrics['f1']:.4f}")
    print(f"ROC-AUC          : {image_metrics['roc_auc']:.4f}")
    print("Confusion matrix (rows = true, cols = predicted):")
    print(image_metrics["confusion_matrix"])
    print("-" * 60)

    if localization_metrics:
        print("LOCALIZATION EVALUATION")
        print("=" * 60)
        print(f"Defective images : {localization_metrics['n_defect_images']}")
        print(f"IoU              : {localization_metrics['iou']:.4f}")
        print(f"Pixel precision  : {localization_metrics['pixel_precision']:.4f}")
        print(f"Pixel recall     : {localization_metrics['pixel_recall']:.4f}")
        print(f"Pixel F1         : {localization_metrics['pixel_f1']:.4f}")
        print("=" * 60)


In [ ]:
# Cell 6 - Create train.py
%%writefile train.py

import argparse

from anomaly_detection import (
    build_autoencoder,
    category_paths,
    check_gpu,
    compute_pixel_threshold,
    compute_threshold,
    describe_architecture,
    load_normal_images,
    plot_training_history,
    save_model,
    save_threshold,
    score_images,
    set_seeds,
    train_autoencoder,
)


def parse_args():
    parser = argparse.ArgumentParser(description="Train the autoencoder")
    parser.add_argument("--category", default="screw",
                        help="MVTec category folder name (screw, wood, bottle, ...)")
    return parser.parse_args()


def main():
    args = parse_args()
    category = args.category
    paths = category_paths(category)

    set_seeds()
    check_gpu()

    # 1. Only normal images: x_train and x_val both come from train/good.
    x_train, x_val = load_normal_images(category)

    # 2. Model.
    model = build_autoencoder()
    describe_architecture(model)

    # 3. Training target is the input image itself.
    history = train_autoencoder(model, x_train, x_val)

    # 4. Save model and thresholds for this category.
    save_model(model, paths["model"])

    val_scores, val_error_maps = score_images(model, x_val)

    threshold_info = compute_threshold(val_scores)
    threshold_info["category"] = category
    threshold_info["pixel_threshold"] = compute_pixel_threshold(val_error_maps)
    threshold_info["n_validation_images"] = int(len(x_val))

    save_threshold(threshold_info, paths["threshold"])

    # 5. Training curves.
    paths["results"].mkdir(parents=True, exist_ok=True)
    plot_training_history(history, paths["results"] / "training_history.png")

    print()
    print(f"Training complete [{category}]")
    print(f"  Model              : {paths['model']}")
    print(f"  Threshold          : {threshold_info['threshold']:.6f}")
    print(f"  Pixel threshold    : {threshold_info['pixel_threshold']:.6f}")
    print(f"  Next               : python evaluate.py --category {category}")


if __name__ == "__main__":
    main()


In [ ]:
# Cell 7 - Create evaluate.py
%%writefile evaluate.py
"""
evaluate.py — evaluate a trained autoencoder on one MVTec test set.

Steps:
    load model + threshold for the category
    load test images (good + all defect types, with ground-truth masks)
    predict every image and compute image-level metrics
    compute pixel-level IoU / precision / recall / F1
    save confusion matrix, ROC curve and example heatmaps

Two threshold protocols:
    python evaluate.py
        threshold from train.py, fitted on the validation split of train/good
    python evaluate.py --domain-calibration 0.5
        threshold fitted on half of the normal TEST images (same capture batch);
        those images are then removed from the evaluation set

Run:
    python evaluate.py --category screw
    python evaluate.py --category wood --domain-calibration 0.5
    python evaluate.py --category bottle --domain-calibration 0.5
"""

import argparse
import json

import numpy as np

from anomaly_detection import (
    anomaly_score,
    calibrate_threshold,
    category_paths,
    evaluate_image_level,
    evaluate_localization,
    load_mask,
    load_model,
    load_test_images,
    load_threshold,
    make_mask,
    plot_confusion_matrix,
    plot_roc_curve,
    print_report,
    reconstruction_error,
    save_threshold,
    set_seeds,
    show_heatmap,
    show_localization,
)


def parse_args():
    parser = argparse.ArgumentParser(description="Evaluate the autoencoder")
    parser.add_argument("--category", default="screw",
                        help="MVTec category folder name (screw, wood, bottle, ...)")
    parser.add_argument("--num-heatmap-examples", type=int, default=4,
                        help="How many normal/defect heatmaps to save")
    parser.add_argument("--skip-localization", action="store_true",
                        help="Skip the pixel-level evaluation")
    parser.add_argument("--domain-calibration", type=float, default=0.0,
                        help="Fraction of normal test images used to fit the "
                             "threshold (e.g. 0.5). Those images are excluded "
                             "from the metrics. 0 keeps the train.py threshold.")
    return parser.parse_args()


def apply_domain_calibration(model, test_images, test_labels, defect_types,
                             mask_paths, fraction, category, threshold_path):
    """Fit the threshold on held-out normal test images."""
    good_index = np.flatnonzero(test_labels == 0)
    n_calibration = int(round(fraction * len(good_index)))

    if n_calibration < 1 or n_calibration >= len(good_index):
        raise ValueError(
            f"--domain-calibration {fraction} would use {n_calibration} of "
            f"{len(good_index)} normal images; choose a fraction that leaves "
            f"at least one normal image for evaluation."
        )

    calibration_index = good_index[:n_calibration]
    keep = np.setdiff1d(np.arange(len(test_labels)), calibration_index)

    print(f"Calibrating on {n_calibration} normal test images; "
          f"evaluating on the remaining {len(keep)} images.")

    info = calibrate_threshold(model, test_images[calibration_index], category)
    save_threshold(info, threshold_path)

    return (
        float(info["threshold"]),
        float(info["pixel_threshold"]),
        test_images[keep],
        test_labels[keep],
        [defect_types[i] for i in keep],
        [mask_paths[i] for i in keep],
    )


def main():
    args = parse_args()
    category = args.category
    paths = category_paths(category)

    set_seeds()

    paths["results"].mkdir(parents=True, exist_ok=True)
    paths["heatmaps"].mkdir(parents=True, exist_ok=True)

    # 1. Load the trained model and the saved threshold.
    model = load_model(paths["model"])
    threshold_info = load_threshold(paths["threshold"])
    threshold = float(threshold_info["threshold"])
    pixel_threshold = float(threshold_info.get("pixel_threshold", threshold))

    # 2. Test data.
    test_images, test_labels, defect_types, mask_paths = load_test_images(category)
    print(f"[{category}] Test set: {len(test_images)} images "
          f"({int(test_labels.sum())} defective, "
          f"{int((test_labels == 0).sum())} normal)")

    # 2b. Optional in-domain calibration.
    if args.domain_calibration > 0:
        (threshold, pixel_threshold, test_images, test_labels,
         defect_types, mask_paths) = apply_domain_calibration(
            model, test_images, test_labels, defect_types, mask_paths,
            args.domain_calibration, category, paths["calibrated_threshold"],
        )
        protocol = "in-domain calibration"
        threshold_source = str(paths["calibrated_threshold"])
    else:
        protocol = "validation split"
        threshold_source = str(paths["threshold"])

    print("Threshold protocol:", protocol)

    # 3. Image-level metrics.
    metrics = evaluate_image_level(model, test_images, test_labels, threshold)

    # 4. Pixel-level metrics.
    localization_metrics = {}
    if not args.skip_localization:
        localization_metrics = evaluate_localization(
            model, test_images, mask_paths, pixel_threshold
        )

    # 5. Report and figures.
    print_report(metrics, localization_metrics, threshold)

    plot_confusion_matrix(metrics["confusion_matrix"],
                          paths["results"] / "confusion_matrix.png")
    plot_roc_curve(metrics["fpr"], metrics["tpr"], metrics["roc_auc"],
                   paths["results"] / "roc_curve.png")

    # 6. Machine-readable summary.
    summary = {
        "category": category,
        "threshold_protocol": protocol,
        "threshold_source": threshold_source,
        "threshold": threshold,
        "pixel_threshold": pixel_threshold,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "roc_auc": metrics["roc_auc"],
        **localization_metrics,
    }
    summary_path = paths["results"] / "metrics_summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=4)

    protocol_name = (
        "metrics_summary_domain_calibration.json"
        if args.domain_calibration > 0
        else "metrics_summary_val_threshold.json"
    )
    protocol_path = paths["results"] / protocol_name
    with open(protocol_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=4)

    print("Saved metrics summary:", summary_path)
    print("Saved protocol summary:", protocol_path)

    # 7. Example heatmaps.
    n_examples = args.num_heatmap_examples
    normal_index = [i for i, y in enumerate(test_labels) if y == 0][:n_examples]
    defect_index = [i for i, y in enumerate(test_labels) if y == 1][:n_examples]

    for i in normal_index + defect_index:
        reconstruction = model.predict(test_images[i:i + 1], verbose=0)[0]
        error_map = reconstruction_error(test_images[i], reconstruction)
        score = anomaly_score(error_map)

        result = {
            "original": test_images[i],
            "reconstruction": reconstruction,
            "error_map": error_map,
            "anomaly_score": score,
            "prediction": "ANOMALY" if score > threshold else "NORMAL",
            "threshold": threshold,
            "predicted_mask": make_mask(error_map, pixel_threshold),
        }

        is_defect = test_labels[i] == 1
        prefix = "defect" if is_defect else "normal"
        name = f"{prefix}_{defect_types[i]}_{i:03d}"

        show_heatmap(
            result,
            save_path=paths["heatmaps"] / f"example_{name}.png",
            title_prefix=f"[{defect_types[i]}] ",
        )

        if is_defect and mask_paths[i] is not None:
            gt_mask = load_mask(mask_paths[i])
            show_localization(
                result,
                gt_mask,
                save_path=paths["heatmaps"] / f"localization_{name}.png",
            )

    print("\nEvaluation complete. Figures are in", paths["results"])


if __name__ == "__main__":
    main()


In [ ]:
# Cell 8 - Create predict.py
%%writefile predict.py
"""
predict.py — predict NORMAL / ANOMALY for one image.

Run:
    python predict.py --image mvtec_anomaly_detection/screw/test/scratch_head/000.png
    python predict.py --image tree.png --category wood --show-heatmap
    python predict.py --image bottle.png --category bottle
    python predict.py --image some_screw.png --threshold-file models/screw_threshold_calibrated.json
"""

import argparse
from pathlib import Path

from anomaly_detection import (
    category_paths,
    load_model,
    load_threshold,
    predict,
    show_heatmap,
)


def parse_args():
    parser = argparse.ArgumentParser(description="Predict one image")
    parser.add_argument("--image", required=True, help="Image to check")
    parser.add_argument("--category", default="screw",
                        help="Which model to use (screw, wood, bottle, ...)")
    parser.add_argument("--threshold-file", default=None,
                        help="Threshold JSON (default: models/<category>_threshold.json)")
    parser.add_argument("--show-heatmap", action="store_true",
                        help="Display the heatmap after prediction")
    parser.add_argument("--save-heatmap", default=None,
                        help="Optional path to save the heatmap PNG")
    return parser.parse_args()


def main():
    args = parse_args()
    paths = category_paths(args.category)

    threshold_path = (
        Path(args.threshold_file) if args.threshold_file else paths["threshold"]
    )

    model = load_model(paths["model"])
    threshold_info = load_threshold(threshold_path)
    threshold = float(threshold_info["threshold"])

    pixel_threshold = threshold_info.get("pixel_threshold")
    if pixel_threshold is not None:
        pixel_threshold = float(pixel_threshold)

    result = predict(model, args.image, threshold, pixel_threshold)

    print("Prediction   :", result["prediction"])
    print("Anomaly Score:", f"{result['anomaly_score']:.4f}")
    print("Threshold    :", f"{result['threshold']:.4f}")
    print("Image        :", result["image_path"])

    if args.show_heatmap or args.save_heatmap:
        save_path = (
            Path(args.save_heatmap)
            if args.save_heatmap
            else paths["heatmaps"] / f"predict_{Path(args.image).stem}.png"
        )
        show_heatmap(result, save_path=save_path, show=args.show_heatmap)


if __name__ == "__main__":
    main()


In [ ]:
# Cell 9 - Imports, seeds and dataset check
import numpy as np
from anomaly_detection import (
    CATEGORY, IMAGE_SIZE, MODEL_PATH, THRESHOLD_PATH,
    CALIBRATED_THRESHOLD_PATH, RESULTS_DIR, HEATMAPS_DIR,
    set_seeds, check_gpu, check_dataset,
    load_normal_images, build_autoencoder, describe_architecture,
    train_autoencoder, save_model, score_images, compute_threshold,
    compute_pixel_threshold, save_threshold, load_test_images,
    evaluate_image_level, evaluate_localization, print_report,
    plot_confusion_matrix, plot_roc_curve, calibrate_threshold,
    reconstruct, reconstruction_error, anomaly_score, make_mask,
    show_heatmap, predict, load_model, load_threshold,
)

set_seeds()
check_gpu()
check_dataset()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
HEATMAPS_DIR.mkdir(parents=True, exist_ok=True)
print('Category:', CATEGORY, '| image size:', IMAGE_SIZE)

In [ ]:
# Cell 10 - Train the autoencoder end-to-end
# Equivalent to: !python train.py

x_train, x_val = load_normal_images()
print('x_train:', x_train.shape, 'x_val:', x_val.shape)

model = build_autoencoder()
describe_architecture(model)

history = train_autoencoder(model, x_train, x_val)
save_model(model, MODEL_PATH)

# Threshold from the normal validation images only
val_scores, val_error_maps = score_images(model, x_val)

threshold_info = compute_threshold(val_scores)
threshold_info['category'] = CATEGORY
threshold_info['pixel_threshold'] = compute_pixel_threshold(val_error_maps)
threshold_info['n_validation_images'] = int(len(x_val))
save_threshold(threshold_info, THRESHOLD_PATH)

print('Image-level threshold:', threshold_info['threshold'])
print('Pixel-level threshold:', threshold_info['pixel_threshold'])

In [ ]:
# Cell 11 - Training curves
from IPython.display import Image, display

plot_training_history(history, RESULTS_DIR / 'training_history.png')
display(Image(filename=str(RESULTS_DIR / 'training_history.png')))

In [ ]:
# Cell 12 - Reconstruct one validation image
import matplotlib.pyplot as plt

sample = x_val[0]
recon = reconstruct(model, sample)
err = reconstruction_error(sample, recon)
score = anomaly_score(err)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample.squeeze(), cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original')
axes[1].imshow(recon.squeeze(), cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Reconstructed')
for ax in axes:
    ax.axis('off')
plt.suptitle(f'Normal validation image - score {score:.4f}')
plt.show()

fig.savefig(RESULTS_DIR / 'reconstruction_example.png', dpi=150,
            bbox_inches='tight')

In [ ]:
# Cell 13 - Evaluate the full test set (validation threshold)
test_images, test_labels, defect_types, mask_paths = load_test_images()
print('Test images:', len(test_images), '| defective:', int(test_labels.sum()))

threshold = float(threshold_info['threshold'])
pixel_threshold = float(threshold_info['pixel_threshold'])

metrics = evaluate_image_level(model, test_images, test_labels, threshold)
loc_metrics = evaluate_localization(model, test_images, mask_paths,
                                    pixel_threshold)
print_report(metrics, loc_metrics, threshold)

plot_confusion_matrix(metrics['confusion_matrix'],
                      RESULTS_DIR / 'confusion_matrix.png')
plot_roc_curve(metrics['fpr'], metrics['tpr'], metrics['roc_auc'],
               RESULTS_DIR / 'roc_curve.png')

In [ ]:
# Cell 14 - Second protocol: calibrate on in-domain normal images
# train/good and test/good come from different captures, so a threshold
# fitted on train/good can be too high for test/good and flag nothing.
# Half of the normal TEST images are used only for calibration and are
# then removed from the evaluation set.
good_index = np.flatnonzero(test_labels == 0)
cal_index = good_index[: len(good_index) // 2]
keep = np.setdiff1d(np.arange(len(test_labels)), cal_index)
print('Calibrating on', len(cal_index), 'normal images; evaluating on', len(keep))

cal_info = calibrate_threshold(model, test_images[cal_index])
save_threshold(cal_info, CALIBRATED_THRESHOLD_PATH)

threshold = float(cal_info['threshold'])
pixel_threshold = float(cal_info['pixel_threshold'])

cal_metrics = evaluate_image_level(model, test_images[keep],
                                   test_labels[keep], threshold)
print_report(cal_metrics, {}, threshold)

In [ ]:
# Cell 15 - Confusion matrix and ROC curve (calibrated threshold)
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name in zip(axes, ['confusion_matrix.png', 'roc_curve.png']):
    ax.imshow(mpimg.imread(RESULTS_DIR / name))
    ax.axis('off')
plt.show()

In [ ]:
# Cell 16 - Anomaly heatmaps for example test images
n_examples = 3
normal_index = [i for i, y in enumerate(test_labels) if y == 0][:n_examples]
defect_index = [i for i, y in enumerate(test_labels) if y == 1][:n_examples]

for i in normal_index + defect_index:
    recon = model.predict(test_images[i:i + 1], verbose=0)[0]
    err = reconstruction_error(test_images[i], recon)
    score = anomaly_score(err)
    result = {
        'original': test_images[i],
        'reconstruction': recon,
        'error_map': err,
        'anomaly_score': score,
        'prediction': 'ANOMALY' if score > threshold else 'NORMAL',
        'threshold': threshold,
        'predicted_mask': make_mask(err, pixel_threshold),
    }
    prefix = 'defect' if test_labels[i] == 1 else 'normal'
    name = f'{prefix}_{defect_types[i]}_{i:03d}'
    show_heatmap(result,
                 save_path=HEATMAPS_DIR / f'example_{name}.png',
                 show=True,
                 title_prefix=f'[{defect_types[i]}] ')

In [ ]:
# Cell 17 - Single-image prediction
sample_defect = str(DATASET_ROOT / 'screw' / 'test' / 'scratch_head' / '000.png')
result = predict(model, sample_defect, threshold, pixel_threshold)
print('Prediction   :', result['prediction'])
print('Anomaly Score:', f"{result['anomaly_score']:.4f}")
print('Threshold    :', f"{result['threshold']:.4f}")

In [ ]:
# Cell 18 - Save/download model + thresholds
from google.colab import files
import shutil

for p in [MODEL_PATH, THRESHOLD_PATH, CALIBRATED_THRESHOLD_PATH,
          RESULTS_DIR / 'training_history.png',
          RESULTS_DIR / 'confusion_matrix.png',
          RESULTS_DIR / 'roc_curve.png']:
    print(' ', p)

shutil.make_archive('screw_anomaly_artifacts', 'zip', '.', base_dir='models')
files.download('screw_anomaly_artifacts.zip')  # comment out to skip

In [ ]:
# Cell 19 - Reload the saved artefacts without retraining
reloaded_model = load_model(MODEL_PATH)
reloaded_threshold = load_threshold(CALIBRATED_THRESHOLD_PATH)
print('Reloaded threshold:', reloaded_threshold)

r = predict(reloaded_model, sample_defect,
            float(reloaded_threshold['threshold']),
            float(reloaded_threshold['pixel_threshold']))
print('Reloaded model:', r['prediction'], '| score', f"{r['anomaly_score']:.4f}")